# សម័យ 1 – ការចាប់ផ្ដើមជជែក (Foundry Local)

កំណត់ត្រាដើមនេះចាប់ផ្ដើម Foundry Local, ទាញយកឈ្មោះអាលីយ៉ាសម៉ូឌែលដែលចូលចិត្ត, និងអនុវត្តទាំងការបញ្ចប់ជជែកទម្រង់ស្តង់ដារ និងការបញ្ចប់ជជែកតាមការស្ទ្រីម។


# សេណារីយ៉ូ
សម័យនេះណែនាំអ្វីដែលតិចបំផុតដែលចាំបាច់ ដើម្បីឲ្យម៉ូដែលភាសាទំហំតូចក្នុងតំបន់ឆ្លើយតបទៅបានតាមរយៈ Foundry Local។ អ្នកនឹង៖
- ដំឡើង SDK / ការពឹងផ្អែករបស់ client។
- ចាប់ផ្តើមម៉េនេជ័រ Foundry Local សម្រាប់អាល់យ៉ាសដែលបានជ្រើស (លំនាំដើម: `phi-4-mini`)។
- អនុវត្ត monkey‑patch ផ្នែកការពារ ដើម្បីទ្រាំវាលជាជម្រើសក្នុងមេតាដាតានៃម៉ូដែល។
- ផ្ញើសំណើបំពេញជជែកស្តង់ដារ។
- ស្ទ្រីមចម្លើយតាមតួកិនមួយៗ។

គោលបំណងគឺដើម្បីផ្ទៀងផ្ទាត់ runtime តំបន់ក្នុងរបស់អ្នក និងផ្លូវបណ្តាញ មុនពេលផ្លាស់ទៅ RAG, routing, ឬ agents។


### ការពន្យល់៖ ការដំឡើងបណ្ណាល័យដែលទាមទារ
ដំឡើងបណ្ណាល័យ Python ដែលចាំបាច់សម្រាប់ចរន្តការជជែកសាមញ្ញនេះ៖
- `foundry-local-sdk`: គ្រប់គ្រងម៉ូដែលក្នុងស្រុក និងរង្វិលជីវិតនៃសេវាកម្ម។
- `openai`: រចនាសម្ព័ន្ធអតិថិជនដែលគេស្គាល់សម្រាប់ការបញ្ចប់សន្ទនា។
- `rich`: សម្រាប់បោះពុម្ពអក្សរឲ្យស្អាត ដើម្បីធ្វើឲ្យលទ្ធផលក្នុងកំណត់ត្រា (notebook) ច្បាស់យាយសម្បូរ។

ការរត់ម្ដងទៀតមានសុវត្ថិភាព (idempotent). សូមរំលង ប្រសិនបើបរិយាកាសរបស់អ្នកមានវារួច។


In [1]:
# Install required libraries (idempotent)
%pip install -q foundry-local-sdk openai rich


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### ការពន្យល់: ការនាំចូលស្នូល
នាំចូលម៉ូឌុលដែលបានប្រើក្នុងសៀវភៅកំណត់ត្រាទាំងមូល:
- `FoundryLocalManager` ដើម្បីធ្វើចំណុចប្រទាក់ជាមួយ runtime ម៉ូឌែលក្នុងស្រុក.
- `OpenAI` client ដើម្បីឲ្យយើងអាចប្រើឡើងវិញផ្ទៃ API chat completion ដែលស្គាល់.
- `rich.print` សម្រាប់លទ្ធផលដែលមានរចនាបថ.

នៅទីនេះមិនមានការហៅបណ្ដាញណាមួយ—វា​គ្រាន់តែរៀបចំដែនឈ្មោះ.


In [2]:
import os
from foundry_local import FoundryLocalManager
from foundry_local.models import FoundryModelInfo
from openai import OpenAI
from rich import print

### ការពិពណ៌នា៖ ការចាប់ផ្តើមអ្នកគ្រប់គ្រង និង ការជួសជុលមេតាដាតា
ធ្វើការចាប់ផ្តើម `FoundryLocalManager` សម្រាប់ឈ្មោះជំនួសដែលបានជ្រើស ហើយអនុវត្ត monkey‑patch ដោយប្រយ័ត្ន ដើម្បីដោះស្រាយយ៉ាងទន់ភ្លន់ចំពោះចម្លើយសេវាកម្មដែល `promptTemplate` អាចមានតម្លៃ `null`។

Key outcomes:
- បញ្ជាក់ពីស្ថានភាពសេវា និង endpoint។
- រាយបញ្ជីម៉ូឌែលដែលបាន cache (ផ្ទៀងផ្ទាត់ឃ្លាំងក្នុងស្រុក)۔
- ដោះស្រាយ ID ម៉ូឌែលជាក់លាក់សម្រាប់ឈ្មោះជំនួស (ប្រើនៅក្នុងការហៅ chat បន្ទាប់)។

បើអ្នកជួបប្រទៈបញ្ហាការផ្ទៀងផ្ទាត់នៅក្នុងមេតាដាតាសេវាកម្មដើម លំនាំនេះបង្ហាញពីរបៀបសម្អាតដោយមិន fork ឬបំបែក SDK។


In [3]:
# Monkeypatch to tolerate service responses where promptTemplate is null
_original_from_list_response = FoundryModelInfo.from_list_response

def _safe_from_list_response(response):  # type: ignore
    try:
        if isinstance(response, dict) and response.get("promptTemplate") is None:
            # Normalize to empty dict so pydantic validation passes
            response["promptTemplate"] = {}
    except Exception as e:  # pragma: no cover
        print(f"[yellow]Warning: safe wrapper encountered issue normalizing promptTemplate: {e}[/yellow]")
    return _original_from_list_response(response)

# Apply patch only once
if getattr(FoundryModelInfo.from_list_response, "__name__", "") != "_safe_from_list_response":
    FoundryModelInfo.from_list_response = staticmethod(_safe_from_list_response)  # type: ignore

ALIAS = os.getenv('FOUNDRY_LOCAL_ALIAS', 'phi-4-mini')
manager = FoundryLocalManager(ALIAS)
print(f'[bold green]Service running:[/bold green] {manager.is_service_running()}')
print(f'Endpoint: {manager.endpoint}')
print('Cached models:', manager.list_cached_models())
model_id = manager.get_model_info(ALIAS).id
print(f'Using model id: {model_id}')

Service running: True

Endpoint: http://127.0.0.1:50262/v1

Cached models:
[
    FoundryModelInfo(
        alias='phi-4-mini',
        id='Phi-4-mini-instruct-generic-gpu:4',
        version='4',
        execution_provider='WebGpuExecutionProvider',
        device_type=<DeviceType.GPU: 'GPU'>,
        uri='azureml://registries/azureml/models/Phi-4-mini-instruct-generic-gpu/versions/4',
        file_size_mb=3809,
        prompt_template={
            'system': '<|system|>{Content}<|end|>',
            'user': '<|user|>{Content}<|end|>',
            'assistant': '<|assistant|>{Content}<|end|>',
            'prompt': '<|user|>{Content}<|end|><|assistant|>'
        },
        provider='AzureFoundry',
        publisher='Microsoft',
        license='MIT',
        task='chat-completion',
        ep_override=None
    ),
    FoundryModelInfo(
        alias='qwen2.5-0.5b',
        id='qwen2.5-0.5b-instruct-generic-gpu:3',
        version='3',
        execution_provider='WebGpuExecutionProvider',
        device_type=<DeviceType.GPU: 'GPU'>,
        uri='azureml://registries/azureml/models/qwen2.5-0.5b-instruct-generic-gpu/versions/3',
        file_size_mb=700,
        prompt_template={
            'system': '<|im_start|>system\n{Content}<|im_end|>',
            'user': '<|im_start|>user\n{Content}<|im_end|>',
            'assistant': '<|im_start|>assistant\n{Content}<|im_end|>',
            'prompt': '<|im_start|>user\n{Content}<|im_end|>\n<|im_start|>assistant'
        },
        provider='AzureFoundry',
        publisher='Microsoft',
        license='apache-2.0',
        task='chat-completion',
        ep_override=None
    ),
    FoundryModelInfo(
        alias='phi-3.5-mini',
        id='Phi-3.5-mini-instruct-generic-gpu:1',
        version='1',
        execution_provider='WebGpuExecutionProvider',
        device_type=<DeviceType.GPU: 'GPU'>,
        uri='azureml://registries/azureml/models/Phi-3.5-mini-instruct-generic-gpu/versions/1',
        file_size_mb=2211,
        prompt_template={
            'prompt': '<|user|>\n{Content}<|end|>\n<|assistant|>',
            'assistant': '<|assistant|>\n{Content}<|end|>'
        },
        provider='AzureFoundry',
        publisher='Microsoft',
        license='MIT',
        task='chat-completion',
        ep_override=None
    )
]

Using model id: Phi-4-mini-instruct-generic-gpu:4

### សេចក្តីពិពណ៌នា: ការបញ្ចប់ជជែកមូលដ្ឋាន
បង្កើត client ដែលផ្គូផ្គងជាមួយ `OpenAI` បញ្ជូនទៅកាន់ endpoint របស់ Foundry ក្នុងស្រុក ហើយអនុវត្តការបញ្ចប់សន្ទនា non‑streaming មួយ។ ផ្តោតនៅទីនេះ:
- ប្រាកដថាម៉ូឌែលឆ្លើយតបដោយគ្មានកំហុស។
- ផ្ទៀងផ្ទាត់ល្បឿន (latency) / ទ្រង់ទ្រាយលទ្ធផល។
- រក្សា `max_tokens` អោយមានតម្លៃទាប ដើម្បីសន្សំធនធាន។

ប្រសិនបើនេះបរាជ័យ សូមពិនិត្យឡើងវិញថាសេវាកម្ម Foundry Local កំពុងដំណើរការ និង alias បានដោះស្រាយយ៉ាងត្រឹមត្រូវ។


In [4]:
client = OpenAI(base_url=manager.endpoint, api_key=manager.api_key or 'not-needed')
prompt = 'List two benefits of local inference for privacy.'
resp = client.chat.completions.create(
    model=model_id,
    messages=[{'role':'user','content':prompt}],
    max_tokens=120,
    temperature=0.5
)
print(resp.choices[0].message.content)

Local inference for privacy refers to the practice of performing data analysis on a local device without sending 
sensitive information to a central server. Two benefits of this approach are:


1. **Enhanced Privacy**: Local inference keeps personal data on the user's device, reducing the risk of data 
breaches and unauthorized access. Since the data is not transmitted over the network, it is less susceptible to 
interception by malicious actors.


2. **Data Sovereignty**: Users retain control over their data, as it does not leave their device. This means that 
individuals or organizations can comply with local data protection regulations, such as the General

### ការពន្យល់៖ ការបញ្ចប់សន្ទនាតាមចរន្ត
បង្ហាញការស្ទ្រីមតូកិន សម្រាប់កាត់បន្ថយអារម្មណ៍ថាពេលយឺត និងពង្រឹងបទពិសោធន៍អ្នកប្រើ (UX). រង្វិលនេះបោះពុម្ពការផ្លាស់ប្តូរបន្ថែមៗជាបន្តបន្ទាប់ពេលវាចូលមក:
- មានប្រយោជន៍សម្រាប់ UI សន្ទនាដែលលទ្ធផលផ្នែកដើមមានសារៈសំខាន់.
- អនុញ្ញាតឱ្យអ្នកវាស់ប្រសិទ្ធភាពហូររបស់តូកិនប្រៀបធៀបនឹងរយៈពេលបញ្ចប់ពេញលេញ.

អ្នកអាចផ្លាស់ប្តូរលំនាំនេះដើម្បីសរុបតូកិន, អាប់ដេតវីជេតបង្ហាញភាពរីកចម្រើន (progress widget), ឬបោះបង់នៅកណ្តាលការបង្កើត.


In [5]:
# Streaming example
stream = client.chat.completions.create(
    model=model_id,
    messages=[{'role':'user','content':'Give a one-sentence definition of edge AI.'}],
    stream=True,
    max_tokens=60,
    temperature=0.4
)
for chunk in stream:
    delta = chunk.choices[0].delta
    if delta and delta.content:
        print(delta.content, end='', flush=True)
print()

Edge

AI

refers

to

artificial

...

latency

and

bandwidth

usage

.

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការមិនទទួលខុសត្រូវ**:
ឯកសារនេះត្រូវបានបកប្រែដោយប្រើសេវាកម្មបកប្រែដោយ AI [Co-op Translator](https://github.com/Azure/co-op-translator). ទោះយ៉ាងណា ខណៈពេលដែលយើងខិតខំរកភាពត្រឹមត្រូវ សូមយកចិត្តទុកដាក់ថា ការបកប្រែស្វ័យប្រវត្តិអាចមានកំហុស ឬភាពមិនត្រឹមត្រូវ។ ឯកសារដើមនៅក្នុងភាសាម្ចាស់ដើមគួរត្រូវបានចាត់ទុកជាធនាគារផ្លូវការ/ប្រភពផ្លូវការ។ សម្រាប់ព័ត៌មានសំខាន់ៗ យើងណែនាំឱ្យប្រើការបកប្រែដោយអ្នកបកប្រែវិជ្ជាជីវៈមនុស្ស។ យើងមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកស្រាយខុសណាមួយដែលកើតឡើងពីការប្រើប្រាស់ការបកប្រែនេះទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
